# Silver Trade History

- **Purpose**: Transforms Bronze TradeHistory into Silver. Type casts IDs to BIGINT, timestamps, and deduplicates.
- **Business Context**: PWG Pipeline - Trade Domain. Tracks trade lifecycle events (PNDG -> SBMT -> CMPT).
- **Execution Frequency**: Per Batch (Full Rebuild from all Bronze)

> Imported our operaitons notebook which include all the functions and all

In [0]:
%run ../../02_common_utils/operations

In [0]:
# configuring widgets for ease of use and reusability
dbutils.widgets.text("env_catalog", "charles_schwab_retailbrokerage_dev_team_lemma")
catalog = dbutils.widgets.get("env_catalog")

bronze_tbl = f"{catalog}.bronze.trade_history"
silver_tbl = f"{catalog}.silver.trade_history"

In [0]:
# Importing all the required functions
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Logging functions for initial load
l_df = spark.sql(f"SELECT * FROM {bronze_tbl} ORDER BY _ingest_ts DESC LIMIT 1")
carried_batch = spark.sql(f"SELECT _batch_id FROM {bronze_tbl} ORDER BY _ingest_ts DESC LIMIT 1").first()[0]
carried_run_id = str(l_df.select("_run_id").first()[0])

log_pipeline_message(spark, carried_run_id, 'INFO', 'silver_trade_history', 'Starting processing for standalone silver trade history CDC updates')

start_pipeline_run(spark, carried_run_id, carried_batch)

log_domain_run_status(spark, carried_run_id, carried_batch, 'TRADE', 'RUNNING')

In [0]:
# Reading the data from the bronze table
df = spark.read.table(bronze_tbl)

# Count of records in the source
source_count = str(df.count())

In [0]:
try:
    # Simple typecasting 
    df2 = df.withColumns({
        "TH_T_ID" : col("TH_T_ID").cast('BIGINT'),
        "TH_DTS"  : to_timestamp(col("TH_DTS"), "yyyy-MM-dd HH:mm:ss")
    })

    # Creating a temp view to use in spark.sql later 
    df2.createOrReplaceTempView('trade_history')
except Exception as e:
    print(f"Typecasting or temp view creation failed: {e}")

In [0]:
try:
    # We are filtering latest rows only based on injesion timestamp
    df3 = spark.sql("""
                    SELECT *, 
                    ROW_NUMBER() OVER(PARTITION BY TH_T_ID, TH_DTS, TH_ST_ID ORDER BY _ingest_ts DESC) AS RN 
                    FROM trade_history
                    """)
    df4 = df3.filter(col("RN") == 1)
except Exception as e:
    print(f"Filtering latest rows failed: {e}")

In [0]:
try:
    # Here we have added new column named load_ts and dropped old metadata columns which are not needed.
    df5 = (
        df4.withColumn("_load_ts", current_timestamp())
        .drop('RN', '_ingest_ts', '_source_file')
    )
except Exception as e:
    print(f"Transforming df4 to df5 failed: {e}")

In [0]:
# Writing the data in to silver table here
df5.write.format("delta").option("overwriteSchema", "true").mode("overwrite").saveAsTable(silver_tbl)

# Checking the final count of the silver table
target_count = spark.read.table(silver_tbl).count()
print(f"Total count of tradehistory is {target_count}")

In [0]:
null_count = spark.sql(f"SELECT COUNT(*) FROM {silver_tbl} WHERE TH_T_ID IS NULL").first()[0]

log_dq_result(spark, carried_run_id, "silver_trade_history", "Null TH_T_ID Check", null_count, source_count)
log_domain_run_status(spark, carried_run_id, carried_batch, 'TRADE', 'COMPLETED')
end_pipeline_run(spark, carried_run_id, 'SUCCESS')
log_pipeline_message(spark, carried_run_id, 'INFO', 'silver_trade_history', 'Successfully completed standalone silver trade history CDC updates')

In [0]:
try:
    # Operations Logging
    # Extract the carry-forwarded _run_id from dataframe
    carried_run_id = str(df5.select("_run_id").first()[0])

    log_pipeline_recon(
        spark=spark,
        run_id=carried_run_id,
        batch_id="ALL",
        domain="TRADE",
        table_name="tradehistory",
        source_layer="bronze",
        target_layer="silver",
        source_count=int(source_count),
        target_count=int(target_count)
    )

    log_audit_event(
        spark=spark,
        run_id=carried_run_id,
        batch="ALL",
        layer="silver",
        table_name="tradehistory",
        operation="OVERWRITE",
        rows_affected=int(target_count)
    )
    print("Done")
except Exception as e:
    print(f"Logging failed: {e}")